# 🧠 Model Training & Evaluation — WM Predictor

Welcome to the **Model Training & Evaluation Notebook**. This notebook demonstrates the core machine learning logic of our World Cup predictor. 

### Key Methodology:
1. **Recency-Weighted Training (Time Decay)**: Matches from years ago are less representative than recent matchups. We compute exponential recency weights $w = e^{-\frac{\Delta t}{1000}}$ to weight recent games higher during model optimization.
2. **Multiclass Outcome Prediction**: We predict the three standard match outcomes: `Home Win (2)`, `Draw (1)`, and `Away Win (0)`.
3. **XGBoost Classifier**: A high-performance Gradient Boosted Decision Tree model.
4. **Hyperparameter Optimization**: Systematic Grid Search across learning rates, depths, estimators, and subsampling rates using 5-fold cross-validation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Set visual style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Training Dataset

We load the engineered training dataset which contains delta features calculated between Team A and Team B (e.g., Difference in total market value, difference in chemistry scores, difference in FIFA rankings, etc.).

In [ ]:
data_path = os.path.join('..', 'data', 'processed', 'model_input', 'training_data.csv')
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])

print(f"✓ Dataset loaded successfully. Total matches: {len(df)}")
df.head()

## 2. Compute Exponential Recency Weights (Time Decay)

To capture current squad dynamics, we discount older games exponentially. A game played 1,000 days (~3 years) ago receives approximately $36.7\%$ of the weight of a game played today.

In [ ]:
latest_date = df['date'].max()
df['days_ago'] = (latest_date - df['date']).dt.days
df['match_weight'] = np.exp(-df['days_ago'] / 1000)

# Plot the time decay function
plt.figure(figsize=(8, 4))
plt.plot(df['days_ago'], df['match_weight'], 'o', alpha=0.3, color='#1f77b4')
plt.title("Exponential Recency Weighting (Time Decay)")
plt.xlabel("Days Ago")
plt.ylabel("Sample Weight")
plt.tight_layout()
plt.show()

## 3. Split Dataset and Define Feature Columns

We specify the model features (the deltas) and split the data into training ($80\%$) and validation/test ($20\%$) sets.

In [ ]:
features = [
    'Delta_Total_Market_Value', 'Delta_Median_Top11_Value', 'Delta_Chemistry',
    'Delta_Form_Rating', 'Delta_UCL_Minutes', 'Delta_Tournament_Minutes',
    'Delta_Average_Age', 'Delta_TM_Value_Rank', 'Delta_FIFA_Rank', 'Delta_FIFA_Points',
    'Is_Neutral' 
]

X = df[features]
y = df['target']
weights = df['match_weight']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} matches")
print(f"Validation set size: {X_test.shape[0]} matches")

## 4. Hyperparameter Tuning via GridSearchCV

We perform systematic cross-validated optimization over our XGBoost hyperparameters to find the ideal combination for predicting outcomes.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'subsample': [0.8, 1.0]
}

xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

print("Starting Hyperparameter Optimization fit...")
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train, sample_weight=w_train)
print(f"🎉 Best Hyperparameters Found: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

## 5. Model Performance Evaluation

We evaluate the optimized model on the held-out validation set and visualize the classification results.

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Optimized Model Test Accuracy: {accuracy * 100:.2f}%")

print("
Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Away Win (0)", "Draw (1)", "Home Win (2)"]))

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Away Win", "Draw", "Home Win"],
            yticklabels=["Away Win", "Draw", "Home Win"])
plt.title("Outcome Prediction Confusion Matrix")
plt.ylabel("Actual Outcome")
plt.xlabel("Predicted Outcome")
plt.tight_layout()
plt.show()

## 6. Feature Importance Visualization

Let's see which metrics carry the most weight in predicting international matchups.

In [ ]:
importances = best_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='#3182bd')
plt.title('Relative Feature Importances (XGBoost)', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Save Optimized Model for Deployment

In [ ]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'xgboost_wm_modelV4.joblib')
joblib.dump(best_model, model_path)
print(f"✓ Optimized model successfully compiled and saved to '{model_path}'!")